In [ ]:
# 🔹 1. Dataset Loading
# Loaded Reddit and Twitter sentiment datasets.
# Standardized column names (text, sentiment).

# 🔹 2. Label Alignment
# Identified common sentiment labels across both platforms.
# Filtered datasets to keep only shared labels.
# Ensured fair cross-platform evaluation.

# 🔹 3. Text Cleaning
# Lowercased text.
# Removed URLs and @mentions.
# Converted hashtags to normal words.
# Removed very short texts.
# Created text_clean column.

# 🔹 4. Class Balancing
# Balanced Reddit (training) dataset.
# Balanced Twitter (test) dataset.
# Capped each class at 1000 samples.
# Ensured fair class distribution.

# 🔹 5. Saved Processed Data
# Saved:
#   reddit_processed.csv
#   twitter_processed.csv
# Made preprocessing reproducible.

# 🔹 6. Domain Analysis
# Calculated average text length (Reddit vs Twitter).
# Visualized length distribution.
# Identified structural differences between platforms.

# 🔹 7. Cross-Platform Experimental Setup
# Split Reddit into:
#   Training set
#   Validation set (stratified)
# Used Twitter as unseen test set.
# Defined experiment:
#   Train on Reddit → Test on Twitter

# 🔹 8. TF-IDF Feature Engineering
# Built TF-IDF features (unigrams + bigrams).
# Limited vocabulary size.
# Removed rare and overly frequent words.
# Prevented data leakage (fit only on training data).

# 🔹 9. Model Training (Baseline Models)
# Trained and compared:
#   Logistic Regression
#   Linear SVM
#   Naive Bayes
#   Random Forest

# Evaluated on:
#   Reddit validation (in-domain)
#   Twitter test (cross-domain)

# Measured:
#   Accuracy
#   Weighted F1
#   Performance drop (generalization gap)
#   Training time

# ================= BASELINE ANALYSIS =================

# 🔹 10. Leaderboard Creation
# Ranked models by Twitter F1 score.

# 🔹 11. Domain Shift Visualization
# Plotted validation vs test performance.
# Visualized generalization gap across models.

# 🔹 12. Confusion Matrix Analysis
# Visualized confusion matrices for best and other models.
# Compared misclassification patterns.

# 🔹 13. Classification Report
# Generated per-class precision, recall, F1.
# Analyzed class-wise weaknesses.

# 🔹 14. Error Analysis
# Extracted misclassified Twitter samples.
# Inspected failure cases manually.
# Identified qualitative domain shift patterns.


# ================= FIX BASELINE =================

# 🔹 15. Model Persistence
# Saved best model (best_model.pkl).
# Saved TF-IDF vectorizer (vectorizer.pkl).
# Saved Logistic Regression separately for calibration.

# 🔹 16. Per-Class Performance Tracking
# Created per_sentiment_performance.csv.
# Stored precision, recall, F1 per class and platform.

# ================= WHY MODEL FAILS =================

# 🔹 17. Linguistic Feature Extraction
# Extracted 10 linguistic features:
#   type-token ratio, sentence count, emoji count,
#   sentiment polarity, etc.

# 🔹 18. Linguistic Compression Analysis
# Compared Reddit vs Twitter linguistic structure.
# Computed compression differences per class.
# Saved compression_by_sentiment.csv.

# 🔹 19. Correlation Analysis
# Measured relationship between linguistic compression and F1 drop.
# Generated compression_vs_transfer.png.
# Identified structural causes of performance degradation.

# 🔹 20. Calibration Analysis
# Used Logistic Regression probabilities.
# Generated calibration curves per class.
# Computed Expected Calibration Error (ECE).
# Showed model overconfidence on cross-domain data.



# ================= FIX + IMPACT =================

# 🔹 21. CORAL Domain Adaptation
# Applied covariance alignment (CORAL) to TF-IDF features.
# Reduced domain mismatch between Reddit and Twitter.
# Trained CORAL-adapted Logistic Regression model.
# Compared baseline vs CORAL performance.
# Saved coral_results.csv and coral_model.pkl.

# 🔹 22. Sentiment Risk Analysis (Mental Health Framing)
# Treated negative sentiment (-1) as high-risk class.
# Computed:
#   F1 drop per class
#   Missed case rate (failure rate)
# Generated:
#   clinical_risk_summary.csv
#   clinical_risk_chart.png
# Highlighted real-world risk of cross-domain failure.

# ================= FINAL INSIGHT =================

# Demonstrated that:
#   - Cross-platform performance drop is caused by linguistic compression.
#   - Models are overconfident on unseen domains.
#   - Domain adaptation (CORAL) can partially mitigate the issue.
#   - Failure in negative sentiment detection poses real-world risk.

In [ ]:
import os

BASE = '/content/sentiment_project'

folders = [
    f'{BASE}/data',
    f'{BASE}/models',
    f'{BASE}/results',
    f'{BASE}/visualisations',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folder structure ready!")

In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

print("Loading datasets...")

# ============ Reddit/Twitter Dataset ============
reddit = pd.read_csv('Reddit_Data.csv')
twitter = pd.read_csv('Twitter_Data.csv')


# Standardize column names
reddit = reddit.rename(columns={'clean_comment': 'text', 'category': 'sentiment'})
twitter = twitter.rename(columns={'clean_text': 'text', 'category': 'sentiment'})


# ============ CLEAN LABELS ============
# Remove rows with missing sentiment labels
reddit = reddit.dropna(subset=['sentiment'])
twitter = twitter.dropna(subset=['sentiment'])

# Convert labels to integer (avoid float/int mismatch)
reddit['sentiment'] = reddit['sentiment'].astype(int)
twitter['sentiment'] = twitter['sentiment'].astype(int)


In [ ]:
# ============ KEEP COMMON EMOTIONS ONLY ============
# Find intersection of sentiment labels shared by both platforms
reddit_labels = set(reddit['sentiment'].unique())
twitter_labels = set(twitter['sentiment'].unique())
common_labels = reddit_labels.intersection(twitter_labels)

print(f"\nReddit sentiment labels: {reddit_labels}")
print(f"Twitter sentiment labels: {twitter_labels}")
print(f"Shared sentiment labels: {common_labels}")

# Keep only rows with shared sentiment labels
reddit = reddit[reddit['sentiment'].isin(common_labels)]
twitter = twitter[twitter['sentiment'].isin(common_labels)]

print(f"\nAfter filtering:")
print(f"Reddit: {len(reddit)} samples")
print(f"Twitter: {len(twitter)} samples")


In [ ]:
# ============ CLEAN TEXT ============
def clean_text(text):
    """Clean text data"""
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # Keep hashtag words
    text = re.sub(r'#(\w+)', r'\1', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

reddit['text_clean'] = reddit['text'].apply(clean_text)
twitter['text_clean'] = twitter['text'].apply(clean_text)

In [ ]:
reddit = reddit[reddit['text_clean'].str.len() > 10]
twitter = twitter[twitter['text_clean'].str.len() > 10]

In [ ]:
# Balance Twitter (test data)
min_samples_twitter = min(twitter['sentiment'].value_counts().min(), 1000)
twitter_balanced = twitter.groupby('sentiment').sample(
    n=min_samples_twitter,
    random_state=42
)


# Balance Reddit (training data)
min_samples_reddit = min(reddit['sentiment'].value_counts().min(), 1000)
reddit_balanced = reddit.groupby('sentiment').sample(
    n=min_samples_reddit,
    random_state=42
)

In [ ]:
# ============ SAVE PROCESSED DATA ============

reddit_balanced.to_csv('reddit_processed.csv', index=False)
twitter_balanced.to_csv('twitter_processed.csv', index=False)

print("\nData preprocessing complete!")
print("\nReddit (Training) distribution:")
print(reddit_balanced['sentiment'].value_counts())
print(f"Total: {len(reddit_balanced)}")

print("\nTwitter (Test) distribution:")
print(twitter_balanced['sentiment'].value_counts())
print(f"Total: {len(twitter_balanced)}")

In [ ]:
import matplotlib.pyplot as plt

# ============ TEXT LENGTH ANALYSIS ============
reddit_balanced['text_length'] = reddit_balanced['text_clean'].str.split().str.len()
twitter_balanced['text_length'] = twitter_balanced['text_clean'].str.split().str.len()

print("\n" + "="*60)
print("PLATFORM COMPARISON")
print("="*60)
print(f"Reddit avg length: {reddit_balanced['text_length'].mean():.1f} words")
print(f"Twitter avg length: {twitter_balanced['text_length'].mean():.1f} words")
print(f"Difference: {reddit_balanced['text_length'].mean() - twitter_balanced['text_length'].mean():.1f} words")

# Visualize length distribution
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist([reddit_balanced['text_length'], twitter_balanced['text_length']],
        bins=30, label=['Reddit', 'Twitter'], alpha=0.7, color=['#FF4500', '#1DA1F2'])
ax.set_xlabel('Text Length (words)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Text Length Distribution: Reddit vs Twitter', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)

# =================================================================================
BASE = '/content/sentiment_project'
FIGS = f'{BASE}/visualisations'

import os
os.makedirs(FIGS, exist_ok=True)

plt.savefig(f'{FIGS}/length_distribution.png', dpi=300, bbox_inches='tight')

print(f"Saved: {FIGS}/length_distribution.png")


# print("Saved: length_distribution.png")

In [ ]:
# train_models.py
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
import pickle
import time

print("="*40)
print("CROSS-PLATFORM EMOTION CLASSIFICATION")
print("Training on REDDIT → Testing on TWITTER")
print("="*40)

# ============ LOAD DATA ============
reddit = pd.read_csv('reddit_processed.csv')
twitter = pd.read_csv('twitter_processed.csv')

# Split Reddit for train/validation
X_train, X_val, y_train, y_val = train_test_split(
    reddit['text_clean'],
    reddit['sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=reddit['sentiment']
)

X_test = twitter['text_clean']
y_test = twitter['sentiment']

print(f"\nDataset sizes---->")
print(f"  Reddit Train: {len(X_train)}")
print(f"  Reddit Val:   {len(X_val)}")
print(f"  Twitter Test: {len(X_test)}")

In [ ]:
# ============ TF-IDF VECTORIZATION ============
print("\nCreating TF-IDF features...")

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.8,
    sublinear_tf=True
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print(f"Feature matrix shape: {X_train_vec.shape}")

In [ ]:
# ============ TRAIN MODELS ============
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    ),

    'SVM': LinearSVC(
        random_state=42,
        max_iter=2000,
        class_weight='balanced'
    ),

    'Naive Bayes': MultinomialNB(alpha=0.1),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
}

results = []

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f">>>> Training {name}...")

    start = time.time()
    model.fit(X_train_vec, y_train)
    train_time = time.time() - start

    # Reddit validation
    y_val_pred = model.predict(X_val_vec)
    val_acc = accuracy_score(y_val, y_val_pred)
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')

    # Twitter test
    y_test_pred = model.predict(X_test_vec)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')

    drop = val_acc - test_acc

    print(f">>>> Trained in {train_time:.1f}s")
    print(f"  Reddit Val:   Acc={val_acc:.3f}, F1={val_f1:.3f}")
    print(f"  Twitter Test: Acc={test_acc:.3f}, F1={test_f1:.3f}")
    print(f"  Drop:         {drop:.3f} ({drop/val_acc*100:.1f}%)")

    results.append({
        'Model': name,
        'Reddit_Val_Acc': round(val_acc, 3),
        'Reddit_Val_F1': round(val_f1, 3),
        'Twitter_Test_Acc': round(test_acc, 3),
        'Twitter_Test_F1': round(test_f1, 3),
        'Acc_Drop': round(drop, 3),
        'Drop_%': round(drop/val_acc*100, 1),
        'Train_Time': round(train_time, 1)
    })




In [ ]:
# NEW ------------------------------------------
# ================= SAVE LOGISTIC REGRESSION =================

import pickle
import os

BASE = '/content/sentiment_project'
MODELS = f'{BASE}/models'

os.makedirs(MODELS, exist_ok=True)

lr_model = models['Logistic Regression']

with open(f'{MODELS}/logistic_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)

print("Saved Logistic Regression model for calibration")

In [ ]:
# ============ RESULTS TABLE ============

results_df = pd.DataFrame(results)

# Sort by Twitter Test F1 (most important metric for cross-domain)
results_df = results_df.sort_values(by="Twitter_Test_F1", ascending=False)

print("\n🏆 MODEL LEADERBOARD (Sorted by Twitter F1)")
display(results_df)

# Save leaderboard
BASE = '/content/sentiment_project'
RES = f'{BASE}/results'

results_df.to_csv(f"{RES}/model_leaderboard.csv", index=False)

print(f"Saved leaderboard to: {RES}/model_leaderboard.csv")


print("\n>>>> Saved: model_leaderboard.csv")

In [ ]:
# ================= SAVE BEST MODEL & VECTORIZER =================

import pickle
import os

BASE = '/content/sentiment_project'
MODELS = f'{BASE}/models'

os.makedirs(MODELS, exist_ok=True)

# Get best model
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print(f"\nSaving best model: {best_model_name}")

# Save model
with open(f'{MODELS}/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save vectorizer
with open(f'{MODELS}/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Saved:")
print(f" - {MODELS}/best_model.pkl")
print(f" - {MODELS}/vectorizer.pkl")

In [ ]:
# ============ DOMAIN DROP VISUALIZATION ============

import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

for _, row in results_df.iterrows():
    plt.plot(
        ["Reddit Val", "Twitter Test"],
        [row["Reddit_Val_F1"], row["Twitter_Test_F1"]],
        marker='o',
        label=row["Model"]
    )

plt.title("Domain Shift: Validation vs Cross-Platform Test", fontsize=14)
plt.ylabel("Weighted F1 Score")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# ============ CONFUSION MATRIX ============

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np

# Get best model (top of leaderboard)
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print(f"\nBest Model: {best_model_name}")

# Predict again (to be safe)
y_test_pred = best_model.predict(X_test_vec)

cm = confusion_matrix(y_test, y_test_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=np.unique(y_test))

fig, ax = plt.subplots(figsize=(6,6))
disp.plot(ax=ax)
plt.title(f"Confusion Matrix (Twitter Test) - {best_model_name}")
plt.show()

In [ ]:
# ============ CONFUSION MATRICES ============

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

best_model_name = results_df.iloc[0]["Model"]

# Collect other models
other_models = [(name, model) for name, model in models.items()
                if name != best_model_name]

print("\n📊 Confusion Matrices (Other Models - Horizontal View)\n")

fig, axes = plt.subplots(1, len(other_models), figsize=(15,4))

for ax, (model_name, model) in zip(axes, other_models):

    y_test_pred_other = model.predict(X_test_vec)
    cm_other = confusion_matrix(y_test, y_test_pred_other)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_other,
        display_labels=np.unique(y_test)
    )

    disp.plot(ax=ax, colorbar=False)
    ax.set_title(model_name, fontsize=10)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ============ CLASSIFICATION REPORT ============

from sklearn.metrics import classification_report

print(f"\n📊 Classification Report (Twitter Test) - {best_model_name}\n")

report = classification_report(y_test, y_test_pred)
print(report)

In [ ]:
# ================= PER-EMOTION PERFORMANCE =================

from sklearn.metrics import classification_report
import pandas as pd
import os

BASE = '/content/sentiment_project'
RES = f'{BASE}/results'

os.makedirs(RES, exist_ok=True)

# ---- Reddit validation predictions ----
y_val_pred = best_model.predict(X_val_vec)

val_report = classification_report(y_val, y_val_pred, output_dict=True)


# ---- Twitter test report ----
test_report = classification_report(y_test, y_test_pred, output_dict=True)

records = []

# Collect per-class sentiment metrics
labels = set(val_report.keys()).union(set(test_report.keys()))

for label in labels:
    if label not in ['accuracy', 'macro avg', 'weighted avg']:

        # Reddit (validation)
        if label in val_report:
            records.append({
                'sentiment': label,
                'platform': 'Reddit',
                'precision': val_report[label]['precision'],
                'recall': val_report[label]['recall'],
                'f1-score': val_report[label]['f1-score']
            })

        # Twitter (test)
        if label in test_report:
            records.append({
                'sentiment': label,
                'platform': 'Twitter',
                'precision': test_report[label]['precision'],
                'recall': test_report[label]['recall'],
                'f1-score': test_report[label]['f1-score']
            })

# Convert to DataFrame
per_sentiment_df = pd.DataFrame(records)

# Save
per_sentiment_df.to_csv(f'{RES}/per_sentiment_performance.csv', index=False)

print("\nSaved:")
print(f"{RES}/per_sentiment_performance.csv")

display(per_sentiment_df.head())

In [ ]:
print("VAL LABELS:", val_report.keys())
print("TEST LABELS:", test_report.keys())

In [ ]:
# ============ ERROR ANALYSIS ============


# Convert to arrays for indexing
X_test_array = np.array(X_test)
y_test_array = np.array(y_test)

wrong_indices = np.where(y_test_pred != y_test_array)[0]

print(f"\n\n          Total misclassified samples: {len(wrong_indices)}")

# Show 20 random errors
np.random.seed(42)
sample_errors = np.random.choice(wrong_indices, size=min(20, len(wrong_indices)), replace=False)

print("\n\n          Sample Misclassifications:          \n")

for idx in sample_errors:
    print("="*80)
    print(f"Text: {X_test_array[idx]}")
    print(f"True Label: {y_test_array[idx]}")
    print(f"Predicted : {y_test_pred[idx]}")

In [ ]:
# For LINGUISTIC COMPRESSION ANALYSIS

import os
import shutil

BASE = '/content/sentiment_project'
DATA = f'{BASE}/data'

os.makedirs(DATA, exist_ok=True)

# Move files
shutil.move('/content/reddit_processed.csv', f'{DATA}/reddit_processed.csv')
shutil.move('/content/twitter_processed.csv', f'{DATA}/twitter_processed.csv')

print("Files moved to project/data folder")

In [ ]:
# For LINGUISTIC COMPRESSION ANALYSIS

# Install and download TextBlob data
!pip install textblob --quiet

import nltk
nltk.download('punkt')
nltk.download('brown')
nltk.download('averaged_perceptron_tagger')

# Download all TextBlob corpora
!python -m textblob.download_corpora

In [ ]:
# ================= LINGUISTIC COMPRESSION ANALYSIS =================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re, os
from textblob import TextBlob
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download once
import nltk
nltk.download('punkt')
nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

BASE = '/content/sentiment_project'
DATA = f'{BASE}/data'
RES = f'{BASE}/results'
FIGS = f'{BASE}/visualisations'

os.makedirs(RES, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

# ---- Load processed data ----
reddit  = pd.read_csv(f'{DATA}/reddit_processed.csv')
twitter = pd.read_csv(f'{DATA}/twitter_processed.csv')

print(f"Reddit: {len(reddit)} rows")
print(f"Twitter: {len(twitter)} rows")


# ================= FEATURE EXTRACTION =================

def extract_linguistic_features(text):
    text = str(text)
    words = text.split()
    wc = max(len(words), 1)

    ttr = len(set(words)) / wc
    avg_word_len = np.mean([len(w) for w in words]) if words else 0
    excl = text.count('!') / wc
    ques = text.count('?') / wc

    emoji_pattern = re.compile('[\U00010000-\U0010ffff]', flags=re.UNICODE)
    emoji_cnt = len(emoji_pattern.findall(text))

    vader = sia.polarity_scores(text)
    pos_r = vader['pos']
    neg_r = vader['neg']

    polarity = TextBlob(text).sentiment.polarity

    first_p = sum(1 for w in words if w.lower() in {'i','me','my','myself','i\'m','i\'ve'})
    fp_density = first_p / wc

    sent_cnt = len(TextBlob(text).sentences)

    return {
        'type_token_ratio': ttr,
        'avg_word_length': avg_word_len,
        'exclamation_dens': excl,
        'question_dens': ques,
        'emoji_count': emoji_cnt,
        'positive_ratio': pos_r,
        'negative_ratio': neg_r,
        'sentiment_polarity': polarity,
        'first_person_dens': fp_density,
        'sentence_count': sent_cnt
    }


# ================= APPLY FEATURES =================

from tqdm.notebook import tqdm
tqdm.pandas()

print("Extracting Reddit features...")
reddit_feats = reddit['text_clean'].progress_apply(lambda t: pd.Series(extract_linguistic_features(t)))
reddit_ling = pd.concat([reddit[['sentiment']], reddit_feats], axis=1)

print("Extracting Twitter features...")
twitter_feats = twitter['text_clean'].progress_apply(lambda t: pd.Series(extract_linguistic_features(t)))
twitter_ling = pd.concat([twitter[['sentiment']], twitter_feats], axis=1)

# Save
reddit_ling.to_csv(f'{DATA}/reddit_linguistic.csv', index=False)
twitter_ling.to_csv(f'{DATA}/twitter_linguistic.csv', index=False)

print("Saved linguistic datasets.")


# ================= COMPRESSION BY EMOTION =================

feat_cols = list(reddit_feats.columns)

reddit_grouped = reddit_ling.groupby('sentiment')[feat_cols].mean().add_prefix('reddit_')
twitter_grouped = twitter_ling.groupby('sentiment')[feat_cols].mean().add_prefix('twitter_')

compression = reddit_grouped.join(twitter_grouped)

for col in feat_cols:
    compression[f'diff_{col}'] = compression[f'reddit_{col}'] - compression[f'twitter_{col}']

compression.to_csv(f'{RES}/compression_by_sentiment.csv')

print("\nCompression summary:")
display(compression.head())


#-----------------------------------------]
summary = compression[[
    'diff_type_token_ratio',
    'diff_sentence_count',
    'diff_sentiment_polarity'
]].copy()

summary.index = summary.index.map({-1:'Negative', 0:'Neutral', 1:'Positive'})

summary.columns = [
    'Vocabulary Change',
    'Sentence Structure Change',
    'Sentiment Shift'
]

display(summary.round(3))


In [ ]:
# ================= CORRELATION WITH F1 DROP =================

per_class_df = pd.read_csv(f'{RES}/per_sentiment_performance.csv')

reddit_f1  = per_class_df[per_class_df['platform']=='Reddit'].set_index('sentiment')['f1-score']
twitter_f1 = per_class_df[per_class_df['platform']=='Twitter'].set_index('sentiment')['f1-score']

f1_drop = (reddit_f1 - twitter_f1).rename('F1_Drop')

plot_df = compression[['diff_type_token_ratio']].join(f1_drop)
plot_df.columns = ['TTR_Diff', 'F1_Drop']
plot_df = plot_df.dropna()

from scipy.stats import pearsonr
r, p = pearsonr(plot_df['TTR_Diff'], plot_df['F1_Drop'])



# Plot - scatter
plt.figure(figsize=(7,5))

plt.scatter(plot_df['TTR_Diff'], plot_df['F1_Drop'], s=120)

for sentiment_label, row in plot_df.iterrows():
    label_map = {-1: 'Negative', 0: 'Neutral', 1: 'Positive'}
    name = label_map.get(sentiment_label, str(sentiment_label))

    plt.annotate(
        name,
        (row['TTR_Diff'], row['F1_Drop']),
        textcoords="offset points",
        xytext=(5,5),
        fontsize=11
    )

plt.axhline(0, linestyle='--', alpha=0.3)
plt.axvline(0, linestyle='--', alpha=0.3)

plt.xlabel('Language Compression (Reddit → Twitter)', fontsize=11)
plt.ylabel('Performance Drop (F1)', fontsize=11)

plt.title(
    f'Language Change vs Model Failure\n(correlation r = {r:.2f})',
    fontsize=13,
    fontweight='bold'
)

plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(f'{FIGS}/compression_vs_transfer.png')
plt.show()


print("---------------------------------------------------------------------------------------------")
print("---------------------------------------------------------------------------------------------")

# Plot - Bar chart
plt.figure(figsize=(7,5))

labels = plot_df.index.map({-1:'Negative', 0:'Neutral', 1:'Positive'})

plt.bar(labels, plot_df['F1_Drop'])

plt.title('Performance Drop by  Class', fontsize=13, fontweight='bold')
plt.ylabel('F1 Drop')
plt.xlabel('Class')

for i, v in enumerate(plot_df['F1_Drop']):
    plt.text(i, v + 0.002, f"{v:.3f}", ha='center')

plt.tight_layout()
plt.savefig(f'{FIGS}/f1_drop_bar.png')
plt.show()

In [ ]:
# ================= CALIBRATION ANALYSIS =================

import pickle
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import label_binarize
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = '/content/sentiment_project'
MODELS = f'{BASE}/models'
DATA = f'{BASE}/data'
RES = f'{BASE}/results'
FIGS = f'{BASE}/visualisations'

# ---- Load model ----
with open(f'{MODELS}/logistic_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open(f'{MODELS}/vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

# ---- Load data ----
twitter = pd.read_csv(f'{DATA}/twitter_processed.csv')

X_test = vectorizer.transform(twitter['text_clean'])
y_test = twitter['sentiment']

# ---- Get probabilities ----
y_prob = model.predict_proba(X_test)
classes = model.classes_

# ---- Binarize labels ----
y_bin = label_binarize(y_test, classes=classes)

ece_results = []

plt.figure(figsize=(12,8))

for i, sentiment_label in enumerate(classes):

    prob_true, prob_pred = calibration_curve(
        y_bin[:, i], y_prob[:, i], n_bins=10
    )

    ece = np.mean(np.abs(prob_true - prob_pred))

    plt.plot(prob_pred, prob_true, marker='o', label=f"{sentiment_label} (ECE={ece:.2f})")

    ece_results.append({
        'sentiment': sentiment_label,
        'ECE': ece
    })

# Perfect line
plt.plot([0,1],[0,1],'k--')

plt.xlabel("Predicted Probability")
plt.ylabel("True Probability")
plt.title("Calibration Curves (Twitter Test)")
plt.legend()

plt.savefig(f'{FIGS}/calibration_curves.png')
plt.show()

# Save results
cal_df = pd.DataFrame(ece_results)
cal_df.to_csv(f'{RES}/calibration_summary.csv', index=False)

print("Saved calibration results.")
display(cal_df)

In [ ]:
# ================= CORAL DOMAIN ADAPTATION =================

import pandas as pd
import numpy as np
import pickle
from scipy.linalg import fractional_matrix_power
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os

BASE = '/content/sentiment_project'
DATA = f'{BASE}/data'
RES = f'{BASE}/results'
MODELS = f'{BASE}/models'

os.makedirs(RES, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

# ---- Load data ----
reddit  = pd.read_csv(f'{DATA}/reddit_processed.csv')
twitter = pd.read_csv(f'{DATA}/twitter_processed.csv')

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    reddit['text_clean'], reddit['sentiment'],
    test_size=0.2, random_state=42, stratify=reddit['sentiment']
)

X_test_raw = twitter['text_clean']
y_test = twitter['sentiment']

# ---- TF-IDF ----
vectorizer = TfidfVectorizer(
    max_features=3000,   # lower to avoid RAM issues
    ngram_range=(1,2),
    min_df=3,
    max_df=0.8,
    sublinear_tf=True
)

vectorizer.fit(X_train_raw)

X_train_sp = vectorizer.transform(X_train_raw)
X_test_sp  = vectorizer.transform(X_test_raw)

# ---- Convert to dense ----
print("Converting to dense...")
Xs = X_train_sp.toarray().astype(np.float64)
Xt = X_test_sp.toarray().astype(np.float64)

print(f"Shapes: {Xs.shape}, {Xt.shape}")

# ---- CORAL FUNCTION ----
def coral_transform(Xs, Xt):
    eps = 1e-3
    Cs = np.cov(Xs, rowvar=False) + eps * np.eye(Xs.shape[1])
    Ct = np.cov(Xt, rowvar=False) + eps * np.eye(Xt.shape[1])

    Ds = fractional_matrix_power(Cs, -0.5)
    Dt = fractional_matrix_power(Ct,  0.5)

    # FIX: remove tiny imaginary parts
    Ds = np.real(Ds)
    Dt = np.real(Dt)

    Xs_aligned = Xs @ Ds @ Dt

    # Also ensure output is real
    Xs_aligned = np.real(Xs_aligned)

    return Xs_aligned


print("Applying CORAL...")
Xs_coral = coral_transform(Xs, Xt)



# ------------------------------------------
# ---- Train baseline ----
lr_base = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_base.fit(X_train_sp, y_train)

base_acc = accuracy_score(y_test, lr_base.predict(X_test_sp))
base_f1  = f1_score(y_test, lr_base.predict(X_test_sp), average='weighted')

# ---- Train CORAL model ----
lr_coral = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_coral.fit(Xs_coral, y_train)

coral_acc = accuracy_score(y_test, lr_coral.predict(Xt))
coral_f1  = f1_score(y_test, lr_coral.predict(Xt), average='weighted')

# ---- Print comparison ----
print("\nRESULTS:")
print(f"Baseline  → Acc: {base_acc:.4f}, F1: {base_f1:.4f}")
print(f"CORAL     → Acc: {coral_acc:.4f}, F1: {coral_f1:.4f}")
print(f"Improvement → {coral_f1 - base_f1:+.4f}")

# ---- Save ----
results = pd.DataFrame([
    {'Model':'Baseline LR', 'Acc':base_acc, 'F1':base_f1},
    {'Model':'CORAL LR', 'Acc':coral_acc, 'F1':coral_f1}
])

results.to_csv(f'{RES}/coral_results.csv', index=False)

with open(f'{MODELS}/coral_model.pkl', 'wb') as f:
    pickle.dump(lr_coral, f)

print("Saved CORAL results.")

In [ ]:
# ================= MENTAL HEALTH RISK ANALYSIS =================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

BASE = '/content/sentiment_project'
RES = f'{BASE}/results'
FIGS = f'{BASE}/visualisations'

os.makedirs(FIGS, exist_ok=True)

# ---- Load per-class sentiment metrics ----
per_class_df = pd.read_csv(f'{RES}/per_sentiment_performance.csv')

reddit_f1  = per_class_df[per_class_df['platform'] == 'Reddit'].set_index('sentiment')['f1-score']
twitter_f1 = per_class_df[per_class_df['platform'] == 'Twitter'].set_index('sentiment')['f1-score']

# ---- Map numeric labels to sentiment names ----
label_map = {
    -1: 'Negative',
     0: 'Neutral',
     1: 'Positive'
}

# Only negative sentiment is treated as clinically/high-risk
clinical_labels = [-1]

records = []

for em in reddit_f1.index:
    rf1 = reddit_f1[em]
    tf1 = twitter_f1[em]

    drop = rf1 - tf1
    missed_rate = drop / rf1 if rf1 > 0 else 0

    records.append({
        'Sentiment_Label': em,
        'Sentiment_Name': label_map.get(em, str(em)),
        'Reddit_F1': rf1,
        'Twitter_F1': tf1,
        'F1_Drop': drop,
        'Missed_Rate': missed_rate,
        'Clinical': em in clinical_labels
    })

risk_df = pd.DataFrame(records).sort_values('F1_Drop', ascending=False)

print("\nSENTIMENT RISK TABLE")
display(risk_df)

# ---- Save ----
risk_df.to_csv(f'{RES}/clinical_risk_summary.csv', index=False)

# ---- Plot ----
plt.figure(figsize=(10, 5))

colors = ['red' if c else 'blue' for c in risk_df['Clinical']]

plt.bar(risk_df['Sentiment_Name'], risk_df['F1_Drop'], color=colors)

plt.title("F1 Drop by Sentiment Class (Negative = High Risk)")
plt.xlabel("Sentiment")
plt.ylabel("F1 Drop")

for i, v in enumerate(risk_df['F1_Drop']):
    plt.text(i, v + 0.002, f"{v:.3f}", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIGS}/clinical_risk_chart.png')
plt.show()

print("Saved sentiment-risk analysis.")

In [ ]:
# Cross-domain drop is real
# Reddit (train) → good performance
# Twitter (test) → worse performance

# Models don’t generalize well across platforms

In [ ]:
# Linguistic Compression
# Twitter = shorter, less expressive
# Reddit = richer, longer

# Model struggles because input style changes

In [ ]:
# tested a solution (CORAL)
# ===================================================================
# Tried to fix domain mismatch
# Result : Performance decreased

# Conclusion:
# Simple statistical alignment is not enough for text data

In [ ]:
# The problem is NOT just distribution shift
# It is also linguistic + semantic shift

In [ ]:
# Future Scope
# ===================================================================
# Use semantic models (BERT) for better understanding
# Apply advanced domain adaptation methods
# Extend to more platforms and finer-grained sentiment label schemes